# EEG Mental Workload Classifier — Phase 1: Data Loading & Exploration

**Dataset:** STEW (Simultaneous Task EEG Workload), from the Kaggle mirror `mitulahirwal/mental-cognitive-workload-eeg-data-stew-dataset`.

Files in this mirror:
- `dataset.mat` — the EEG signal data
- `rating.mat` — raw subjective workload ratings (1-9 scale)
- `class_012.mat` — 3-class labels (integer-encoded: 0/1/2 = low/moderate/high, per the standard STEW convention of splitting 1-3 / 4-6 / 7-9)
- `three_class_one_hot.mat` — same 3-class labels, one-hot encoded
- `EEG data summary.pdf` — documentation (not loaded in code, worth reading once)

**Goal:** Classify mental workload level from short EEG recordings. We'll start with the 3-class labels and can collapse to binary (high vs. low, dropping/merging moderate) later if that gives a cleaner signal -- this mirrors the standard STEW binary framing used in most published baselines.

**Why this matters (psych/neuro framing):** Mental workload is a core construct in cognitive load theory -- as task demands increase, EEG shows measurable shifts (typically increased frontal theta power and decreased parietal alpha power). This notebook explores whether those signatures are recoverable from the raw signal and sets up the pipeline for feature extraction + modeling in later notebooks.

**Pipeline overview:**
1. Load the four `.mat` files
2. Inspect structure and cross-check that sample counts line up across files
3. Visualize sample EEG traces and basic spectral content
4. Save a clean, checkpointed version to Drive for downstream notebooks

> Run this in Google Colab. First cell mounts Drive so we don't lose downloaded/processed data between sessions.

## 0. Environment setup

In [ ]:
# We deliberately do NOT use the Hugging Face `datasets` library here --
# it fails on this data due to a library-side breaking change unrelated to us.
# Loading .mat files directly with scipy avoids that dependency entirely.
!pip install -q mne scipy scikit-learn matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/eeg-workload-project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)
print('Project dir:', PROJECT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from scipy.io import loadmat

sns.set_theme(style='whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Upload the Kaggle download

Download the dataset zip from the Kaggle page (Download button, top right), then run the cell below and select that zip file. We'll unzip it into `data/raw/` on Drive so it persists across sessions.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # select the .zip downloaded from Kaggle

for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall(f'{PROJECT_DIR}/data/raw')
        print(f'Unzipped {fname} into {PROJECT_DIR}/data/raw')
    else:
        # in case individual files were uploaded instead of a zip
        dest = f'{PROJECT_DIR}/data/raw/{fname}'
        with open(dest, 'wb') as f:
            f.write(uploaded[fname])
        print('Saved', dest)

In [ ]:
# Confirm what's actually in data/raw (and any subfolders from the zip extraction)
for root, dirs, filenames in os.walk(f'{PROJECT_DIR}/data/raw'):
    for fn in filenames:
        print(os.path.join(root, fn))

## 2. Load and inspect all four `.mat` files

**Adjust `RAW_DATA_DIR` below** if the file listing above shows the files nested inside a subfolder rather than directly in `data/raw/`.

In [ ]:
RAW_DATA_DIR = f'{PROJECT_DIR}/data/raw'  # adjust if files are nested in a subfolder

MAT_FILES = {
    'dataset': 'dataset.mat',
    'rating': 'rating.mat',
    'class_012': 'class_012.mat',
    'three_class_one_hot': 'three_class_one_hot.mat',
}

mat_data = {}
for label, fname in MAT_FILES.items():
    path = os.path.join(RAW_DATA_DIR, fname)
    contents = loadmat(path)
    data_keys = [k for k in contents.keys() if not k.startswith('__')]
    mat_data[label] = contents
    print(f'--- {fname} ---')
    for k in data_keys:
        arr = contents[k]
        print(f'  {k}: shape {arr.shape}, dtype {arr.dtype}')
    print()

**Stop here and check the printed output before continuing.** You should see one non-metadata variable per file (or a small number). Confirm:
- `dataset.mat`'s array shape -- expect something like `(n_samples, n_channels, n_timepoints)` or a flattened 2D version `(n_samples, n_channels * n_timepoints)`
- `rating.mat` and `class_012.mat` should have a length matching `n_samples` from `dataset.mat` (or `n_samples` per subject/condition -- STEW has 48 subjects x 2 conditions = 96 raw recordings before any windowing)

If a shape doesn't look like what's described above, paste it back before running the next cell -- the variable-name extraction below assumes standard single-variable-per-file `.mat` exports.

In [ ]:
# Extract the actual data array from each loaded .mat dict.
# Assumes each file has exactly one non-metadata variable -- adjust the
# key selection (e.g. pick a specific key by name) if a file has more than one.

def get_sole_variable(mat_dict):
    data_keys = [k for k in mat_dict.keys() if not k.startswith('__')]
    if len(data_keys) != 1:
        raise ValueError(f'Expected exactly one data variable, found {data_keys} -- inspect manually')
    return mat_dict[data_keys[0]]

X_raw = get_sole_variable(mat_data['dataset'])
ratings_raw = get_sole_variable(mat_data['rating'])
class_012 = get_sole_variable(mat_data['class_012']).squeeze()
class_one_hot = get_sole_variable(mat_data['three_class_one_hot'])

print('X_raw shape:', X_raw.shape)
print('ratings_raw shape:', ratings_raw.shape)
print('class_012 shape:', class_012.shape, '| unique values:', np.unique(class_012))
print('class_one_hot shape:', class_one_hot.shape)

## 3. Reshape EEG data into `(n_samples, n_channels, n_timepoints)` if needed

STEW's standard format is 14 channels at 128Hz. If `X_raw` came in flattened (2D), reshape it here. **Only run the reshape if `X_raw.ndim == 2`** -- if it's already 3D, skip straight to using `X_raw` as `X`.

In [ ]:
N_CHANNELS = 14
SFREQ = 128  # Hz
CHANNEL_NAMES = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']

if X_raw.ndim == 2:
    n_samples = X_raw.shape[0]
    n_timepoints = X_raw.shape[1] // N_CHANNELS
    print(f'Attempting reshape: {X_raw.shape} -> ({n_samples}, {N_CHANNELS}, {n_timepoints})')
    # NOTE: confirm whether the flattening order is channel-major or time-major
    # before trusting this reshape -- check EEG data summary.pdf, or verify by
    # plotting a sample and checking it looks like a coherent EEG trace (Section 5).
    X = X_raw.reshape(n_samples, N_CHANNELS, n_timepoints)
else:
    X = X_raw
    print('X_raw already 3D, using as-is. Shape:', X.shape)

y = class_012  # 3-class integer labels: 0=low, 1=moderate, 2=high (confirm against summary PDF)
print('Final X shape:', X.shape)
print('Final y shape:', y.shape)

## 4. Visualize a sample EEG epoch

Sanity-check the signal looks like real EEG (coherent oscillatory waveform, reasonable amplitude, not noise) -- this is also how we'll confirm the reshape above used the correct axis order.

In [ ]:
def plot_sample_epoch(X, idx, y=None, sfreq=SFREQ, channel_names=CHANNEL_NAMES):
    sample = X[idx]
    n_channels = sample.shape[0]
    t = np.arange(sample.shape[1]) / sfreq

    fig, axes = plt.subplots(n_channels, 1, figsize=(10, 1.2 * n_channels), sharex=True)
    for i, ax in enumerate(axes):
        ax.plot(t, sample[i], linewidth=0.6)
        ax.set_ylabel(channel_names[i] if i < len(channel_names) else f'ch{i}', rotation=0, ha='right', fontsize=8)
        ax.set_yticks([])
    axes[-1].set_xlabel('Time (s)')
    title = f'Sample epoch #{idx}'
    if y is not None:
        title += f' — class label: {y[idx]}'
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/figures/sample_epoch_{idx}.png', dpi=150)
    plt.show()

plot_sample_epoch(X, 0, y=y)

## 5. Quick spectral check

Confirm plausible EEG spectral content (power concentrated below ~45Hz) before investing time in the full MNE preprocessing pipeline (Notebook 2).

In [ ]:
from scipy.signal import welch

def plot_psd(X, idx, y=None, sfreq=SFREQ, channel_names=CHANNEL_NAMES):
    sample = X[idx]
    fig, ax = plt.subplots(figsize=(8, 5))
    for i in range(sample.shape[0]):
        freqs, psd = welch(sample[i], fs=sfreq, nperseg=min(256, sample.shape[1]))
        ax.semilogy(freqs, psd, label=channel_names[i] if i < len(channel_names) else f'ch{i}', linewidth=0.8)
    ax.set_xlim(0, 50)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD (log scale)')
    title = f'Power spectral density — epoch #{idx}'
    if y is not None:
        title += f', class label: {y[idx]}'
    ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/figures/psd_epoch_{idx}.png', dpi=150)
    plt.show()

plot_psd(X, 0, y=y)

## 6. Class balance

STEW does not appear to include a subject-ID array in this particular mirror's files, based on what we've seen so far -- if `n_samples` in `dataset.mat` doesn't cleanly map to 48 subjects (e.g. it's already windowed into many more short epochs), we won't be able to do subject-level grouping unless the summary PDF documents an implicit ordering (e.g. samples grouped sequentially by subject). **Check `EEG data summary.pdf` for this** before Notebook 2 -- subject-independent splitting is important for valid evaluation, so it's worth resolving before modeling.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(y).value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Class balance (0=low, 1=moderate, 2=high)')
ax.set_ylabel('Count')
ax.set_xlabel('Class')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/figures/class_balance.png', dpi=150)
plt.show()

print(pd.Series(y).value_counts(normalize=True).sort_index())

## 7. Checkpoint: save cleaned arrays

Save as `.npy` so we never have to re-load/re-parse the `.mat` files in later notebooks -- avoids losing work to Colab session timeouts.

In [ ]:
np.save(f'{PROJECT_DIR}/data/processed/X_raw.npy', X)
np.save(f'{PROJECT_DIR}/data/processed/y_class012.npy', y)
np.save(f'{PROJECT_DIR}/data/processed/ratings_raw.npy', ratings_raw)

print('Checkpoint saved to', f'{PROJECT_DIR}/data/processed/')

## Next steps (Notebook 2)

1. Resolve subject-grouping question (check the summary PDF) so we can do subject-independent cross-validation
2. Bandpass filter (4-45 Hz) + artifact handling via MNE
3. Extract band-power features (theta, alpha, beta, gamma) per channel -- this is where cognitive load theory informs feature choice: expect frontal theta increase + parietal alpha decrease under high workload
4. Baseline models: Random Forest / SVM / XGBoost on engineered features
5. Decide: keep 3-class, or collapse to binary (matching the standard published STEW binary framing) for a cleaner first pass